In [13]:
%%writefile HPC_2B.cpp
#include <iostream>
#include <omp.h>

using namespace std;

void merge(int a[], int l, int m, int r) {
    int temp[100];
    int i = l, j = m + 1, k = 0;

    while (i <= m && j <= r)
        temp[k++] = (a[i] < a[j]) ? a[i++] : a[j++];

    while (i <= m)
        temp[k++] = a[i++];

    while (j <= r)
        temp[k++] = a[j++];

    for (i = l, k = 0; i <= r; i++, k++)
        a[i] = temp[k];
}

// Sequential Merge Sort
void seqMergeSort(int a[], int l, int r) {
    if (l < r) {
        int m = (l + r) / 2;

        seqMergeSort(a, l, m);
        seqMergeSort(a, m + 1, r);

        merge(a, l, m, r);
    }
}

// Parallel Merge Sort
void parMergeSort(int a[], int l, int r) {
    if (l < r) {
        int m = (l + r) / 2;

        #pragma omp parallel sections
        {
            #pragma omp section
            parMergeSort(a, l, m);

            #pragma omp section
            parMergeSort(a, m + 1, r);
        }

        merge(a, l, m, r);
    }
}

int main() {
    int n;
    cout << "Enter number of elements: ";
    cin >> n;

    int a[100], b[100];
    cout << "Enter elements:\n";
    for (int i = 0; i < n; i++) {
        cin >> a[i];
        b[i] = a[i];
    }

    // Sequential Time
    double start = omp_get_wtime();

    seqMergeSort(a, 0, n - 1);

    double end = omp_get_wtime();

    cout << "Sequential Time: " << end - start << endl;

    // Parallel Time
    start = omp_get_wtime();

    parMergeSort(b, 0, n - 1);

    end = omp_get_wtime();

    cout << "Parallel Time: " << end - start << endl;

    cout << "Sorted Array:\n";

    for (int i = 0; i < n; i++)
        cout << b[i] << " ";

    return 0;
}

Overwriting HPC_2B.cpp


In [14]:
!g++ -fopenmp HPC_2B.cpp -o HPC_2B

In [15]:
!./HPC_2B

Enter number of elements: 5
Enter elements:
1 4 2 3 5
Sequential Time: 1.24e-06
Parallel Time: 0.000191859
Sorted Array:
1 2 3 4 5 